###  Inject Azure Authentication from your shared folder

### # =============================================================
### # nb_raw_to_bronze.py
### # Layer     : Bronze
### # Purpose   : Orchestrator only. Coordinates the 4 specialist
### #             notebooks. Contains zero business logic itself.
### #
### # Called by : ADF Notebook Activity (inside ForEach loop)
### # Calls     : set_auth_context → schema_manager →
### #             autoloader → lineage → writer
### #
### # ADF passes these as base parameters:
### #   table_name    : e.g. "device_snapshots"
### #   table_type    : "dimension" or "fact"
### #   load_date     : "yyyy-mm-dd"  (from Tumbling Window trigger)
### #   storage_account: "petiot"
### #   is_incremental: "true" or "false"
### # =============================================================

In [0]:
# clearing the watermark table:
# dbutils.fs.rm("abfss://checkpoints@petiot.dfs.core.windows.net/bronze/device_snapshots/", recurse=True)

In [0]:
# ── CELL 1: Widgets (receive ADF parameters) ─────────────────
# Do not remove injected widgets here; ADF/Jobs pass runtime values through them.
WIDGET_DEFAULTS = {
    "p_table_name": "device_snapshots",
    "p_table_type": "fact",
    "p_window_start": "2025-01-01T00:00:00Z",
    "storage_account": "petiot",
    "is_incremental": "true",
}


def _get_widget_if_present(name: str) -> str | None:
    try:
        value = dbutils.widgets.get(name).strip()
        return value or None
    except Exception:
        return None


def _require_widget(name: str) -> str:
    value = _get_widget_if_present(name)
    if value is None:
        dbutils.widgets.text(name, WIDGET_DEFAULTS[name])
        value = dbutils.widgets.get(name).strip()
    if not value:
        raise ValueError(f"Widget '{name}' must be provided.")
    return value


def _resolve_widget_value(*names: str, default: str) -> str:
    for name in names:
        value = _get_widget_if_present(name)
        if value is not None:
            return value

    primary_name = names[0]
    dbutils.widgets.text(primary_name, default)
    value = dbutils.widgets.get(primary_name).strip()
    if not value:
        raise ValueError(f"One of widgets {names} must be provided.")
    return value


storage_account_value = _resolve_widget_value("storage_account", "p_storage_account", default=WIDGET_DEFAULTS["storage_account"])
is_incremental_value = _resolve_widget_value("is_incremental", "p_is_incremental", default=WIDGET_DEFAULTS["is_incremental"])

# Preserve backward compatibility for any downstream code that reads canonical widget names.
if _get_widget_if_present("storage_account") is None:
    dbutils.widgets.text("storage_account", storage_account_value)
if _get_widget_if_present("is_incremental") is None:
    dbutils.widgets.text("is_incremental", is_incremental_value)

RUN_CONTEXT = {
    "table_name": _require_widget("p_table_name"),
    "table_type": _require_widget("p_table_type").lower(),
    "window_start": _require_widget("p_window_start"),
    "storage_account": storage_account_value,
    "is_incremental": is_incremental_value.lower() == "true",
}

# Keep these aliases for downstream notebooks/cells that expect them.
TABLE_NAME = RUN_CONTEXT["table_name"]
TABLE_TYPE = RUN_CONTEXT["table_type"]
WINDOW_START = RUN_CONTEXT["window_start"]
STORAGE_ACCOUNT = RUN_CONTEXT["storage_account"]
IS_INCREMENTAL = RUN_CONTEXT["is_incremental"]

# Format window_start to act as your LOAD_DATE (Extracting YYYY-MM-DD from timestamp)
# Example: "2026-06-17T18:30:00Z" -> "2026-06-17"
LOAD_DATE = WINDOW_START[:10]
RUN_CONTEXT["load_date"] = LOAD_DATE

print("=" * 60)
print("   nb_raw_to_bronze")
print(f"   Table         : {TABLE_NAME}")
print(f"   Type          : {TABLE_TYPE}")
print(f"   Window Start  : {WINDOW_START}")
print(f"   Load Date     : {LOAD_DATE}")
print(f"   Incremental   : {IS_INCREMENTAL}")
print(f"   Storage       : {STORAGE_ACCOUNT}")
print("=" * 60)

In [0]:
# ── CELL 2: Shared helpers ───────────────────────────────────

def log_step(message: str) -> None:
    print(f"[nb_raw_to_bronze] {message}")


def abfss_path(container: str, path: str = "", storage_account: str = None) -> str:
    active_storage_account = storage_account or STORAGE_ACCOUNT
    clean_path = path.lstrip("/")
    base = f"abfss://{container}@{active_storage_account}.dfs.core.windows.net"
    return f"{base}/{clean_path}" if clean_path else f"{base}/"


def abfss(container: str, path: str = "", storage_account: str = None) -> str:
    return abfss_path(container, path, storage_account)


def ensure_auth_context() -> None:
    auth_type_key = f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net"
    if spark.conf.get(auth_type_key, None):
        log_step("Azure auth context already available.")
        return

    log_step("Azure auth context missing. Loading shared auth notebook.")
    get_ipython().run_line_magic("run", "/Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb")
    log_step("Azure auth context loaded.")

In [0]:
# ── CELL 3: Inspect Auto Loader state ─────────────────────────
checkpoint_path = abfss_path("checkpoints", f"bronze/{TABLE_NAME}/")
offset_path = f"{checkpoint_path}offsets/"

# List the internal offsets directory where Auto Loader stores file states
try:
    display(dbutils.fs.ls(offset_path))
except Exception:
    log_step(f"Offsets directory does not exist yet for '{TABLE_NAME}'. First load or watermark not present.")

In [0]:
# ── CELL 4: Ensure Azure auth context ─────────────────────────
ensure_auth_context()

In [0]:
# ── CELL 3: Load specialist notebooks ────────────────────────
# Each %run injects its functions into this notebook's scope.
# Order matters: schema_manager first, then autoloader uses abfss.
%run /Workspace/Shared/pet-analytics/bronze/schema_manager.py.ipynb
%run /Workspace/Shared/pet-analytics/bronze/autoloader.py.ipynb
%run /Workspace/Shared/pet-analytics/bronze/lineage.py.ipynb
%run /Workspace/Shared/pet-analytics/bronze/writer.py.ipynb

In [0]:
# ── CELL 4: Get schema ────────────────────────────────────────
# schema_manager owns StructType — bronze_loader just asks for it
schema = get_schema(TABLE_NAME)
print(f"\n[bronze_loader] Schema retrieved for '{TABLE_NAME}'")
print(f"  Columns: {[f.name for f in schema.fields]}")

In [0]:
# ── CELL 5: Read data ─────────────────────────────────────────
# Route: streaming (fact + incremental) vs batch (dimension / backfill)
ensure_auth_context()

if TABLE_TYPE == "fact" and IS_INCREMENTAL:
    # Auto Loader streaming — tracks files via checkpoint
    # Will only pick up files not yet processed
    checkpoint_path = build_checkpoint_path(TABLE_NAME)
    source_path = build_source_path(TABLE_NAME, TABLE_TYPE)
    schema_location = build_schema_location(TABLE_NAME)
    df = (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.schemaLocation", schema_location)
             .option("cloudFiles.inferColumnTypes", "false")
             .option("multiLine", "true")
             .option("recursiveFileLookup", "true")
             .schema(schema)
             .load(source_path)
    )
    log_step(f"Stream source    : {source_path}")
    log_step(f"Schema location  : {schema_location}")
    log_step("Mode             : STREAMING (Auto Loader)")
else:
    # Batch — for dimensions or forced full refresh
    df = read_batch(TABLE_NAME, TABLE_TYPE, schema, LOAD_DATE)
    log_step("Mode             : BATCH (full refresh)")

In [0]:
# ── CELL 6: Schema validation ─────────────────────────────────
# Validates incoming schema against registry.
# Extra columns are allowed (schema evolution).
# Missing columns or type mismatches raise RuntimeError.
validate_schema(TABLE_NAME, df)
print(f"[bronze_loader] Schema validation passed.")

In [0]:
# ── CELL 7: Add lineage columns ───────────────────────────────
# lineage.py injects audit columns — no business logic
df = add_lineage_columns(df, TABLE_NAME, LOAD_DATE, TABLE_TYPE)
print(f"[bronze_loader] Lineage columns added.")

In [0]:
# ── CELL 8: Write to Bronze Delta ─────────────────────────────
# writer.py handles all Delta write logic
if TABLE_TYPE == "fact" and IS_INCREMENTAL:
    write_stream(df, TABLE_NAME, checkpoint_path)
else:
    write_batch(df, TABLE_NAME)

In [0]:
# ── CELL 9: Confirm ───────────────────────────────────────────
row_count = get_bronze_row_count(TABLE_NAME)
RUN_CONTEXT["row_count"] = row_count

print(f"\n{'=' * 60}")
print("  Bronze write complete")
print(f"  Table     : {TABLE_NAME}")
print(f"  Load date : {LOAD_DATE}")
print(f"  Rows in Bronze Delta: {row_count:,}")
print(f"{'=' * 60}")


In [0]:
# Return to ADF — can be used in subsequent pipeline activities
result_message = f"SUCCESS|{TABLE_NAME}|{RUN_CONTEXT['row_count']}"
log_step(f"Notebook exit payload: {result_message}")
dbutils.notebook.exit(result_message)